# SecuMail 2.0

Exécutez la cellule **0** au début de chaque séance, puis la section du jour.
Le code à modifier est dans le dossier `homework/`.

| Séance | Section | Fichier à modifier |
|---|---|---|
| 1 | 1. L'ancienne passerelle | aucun |
| 2 | 3. Module 1 | `homework/phishing_rules.py` |
| 3 | 4. Module 2, 5. Mesurer | `homework/spam_rules.py` |
| 4 | 6. Module 3 | `homework/bayes.py` |
| 5 | 7. Calibrer, 8. Remise | `homework/thresholds.py` et les `POINTS_` |

## 0. Démarrage

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    projet = Path("/content/drive/MyDrive/secumail")
    if projet.exists():
        os.chdir(projet)
except ImportError:
    pass

RACINE = Path.cwd()
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

%load_ext autoreload
%autoreload 2

from src import evaluation, ml, pipeline, scoring_engine, verification

table_marques = pipeline.charger_domaines_officiels()
courriels = pipeline.charger_courriels()
entrainement, validation = pipeline.separer_validation(courriels)
reels = [c["label"] for c in validation]

print(f"{len(table_marques)} marques connues")
print(f"{len(entrainement)} courriels pour apprendre et régler")
print(f"{len(validation)} courriels pour mesurer (validation)")

---
## 1. L'ancienne passerelle

Voici tout ce qui reste après l'incident : chaque courriel reçoit un verdict
tiré au hasard. Mesurons-la sur les courriels de validation.

In [ ]:
ancienne = evaluation.rapport(
    reels, scoring_engine.ancienne_passerelle(validation), "Ancienne passerelle"
)

Notez le **coût total**. C'est le chiffre que votre équipe doit battre.
Chaque erreur coûte des points, et les erreurs graves coûtent plus cher :

| Erreur | Coût |
|---|---|
| hameçonnage livré (verdict HAM) | 8 |
| courriel légitime bloqué comme PHISHING | 10 |
| courriel légitime classé SPAM | 5 |
| courriel légitime mis en quarantaine | 3 |
| spam livré | 1 |

---
## 2. Vérification

Relancez cette cellule chaque fois que vous modifiez un fichier de `homework/`.
`OK` : réussi. `...` : pas encore fait. `X` : erreur à corriger.

In [ ]:
verification.run();

---
## 3. Module 1 : l'identité de l'expéditeur

Fichier : `homework/phishing_rules.py`. Commencez par le fragment survivant,
`domaine_est_officiel()` : la vérification montre qu'il a une faille.

Le rapport ci-dessous montre les indices trouvés pour un courriel.
Changez `numero` pour en examiner d'autres.

In [ ]:
numero = 0
courriel = validation[numero]
print("Étiquette réelle :", courriel["label"].upper())
scoring_engine.afficher_rapport(
    scoring_engine.analyser_courriel(courriel, table_marques)
)

---
## 4. Module 2 : le contenu du message

Fichier : `homework/spam_rules.py`. Vos listes de mots et de motifs doivent
venir des courriels. Lisez quelques spams d'entraînement pour les trouver.

In [ ]:
import random

spams = [c for c in entrainement if c["label"] == "spam"]
for c in random.sample(spams, 5):
    print(f"--- {c['subject']}\n{c['message']}\n")

Les courriels qui ont le score de spam le plus élevé :

In [ ]:
resultats = scoring_engine.analyser_tous(validation, table_marques)
for r in sorted(resultats, key=lambda r: r["score_spam"], reverse=True)[:5]:
    c = r["courriel"]
    print(f"{r['score_spam']:>3}/100  [{c['label']}]  {c['subject'][:60]}")
    for points, explication in r["details_spam"]:
        print(f"         +{points:<3} {explication}")

---
## 5. Mesurer le système à règles

Les deux modules et le moteur de scoring, sur les courriels de validation.
Puis les courriels mal classés : la matière première de l'analyse d'erreurs.

In [ ]:
resultats = scoring_engine.analyser_tous(validation, table_marques)
verdicts = [r["verdict"] for r in resultats]
regles = evaluation.rapport(reels, verdicts, "Règles (modules 1 et 2)")
evaluation.montrer_erreurs(validation, reels, verdicts);

---
## 6. Module 3 : le module qui apprend

Fichier : `homework/bayes.py`. Le module apprend sur les courriels
d'entraînement, puis on le mesure sur les **mêmes** courriels de validation
que les règles.

In [ ]:
table, totaux = ml.entrainer(entrainement)
print(f"{len(table)} mots appris")

predictions = ml.predire_tous(validation, table, totaux)
appris = evaluation.rapport(reels, predictions, "Module 3 (appris)")
evaluation.montrer_erreurs(validation, reels, predictions);

In [ ]:
for categorie in ("spam", "phishing", "ham"):
    print(f"\nMots qui pèsent le plus pour : {categorie}")
    for mot, points in ml.mots_les_plus_forts(table, totaux, categorie, 8):
        print(f"  {mot:<20} {points:+.1f}")

In [ ]:
numero = 0
courriel = validation[numero]
print(f"{courriel['subject']}  (réel : {courriel['label']})")
ml.afficher_prediction(courriel, table, totaux)

---
## 7. Comparer et calibrer

Vos deux systèmes sur les mêmes courriels, avec la passerelle au hasard comme point de départ :

In [ ]:
evaluation.comparer(ancienne, regles, appris);

Votre système à règles peut faire mieux. Deux leviers :

- les `SEUIL_` dans `homework/thresholds.py`
- les `POINTS_` dans `homework/phishing_rules.py` et `homework/spam_rules.py`

Chaque courbe montre le coût total quand on change un seul seuil. Changez une
valeur à la fois, relancez les sections 5 et 7, et notez le coût avant et
après dans votre cahier.

In [ ]:
for nom in [
    "SEUIL_PHISHING",
    "SEUIL_PHISHING_SUSPECT",
    "SEUIL_SPAM",
    "SEUIL_SPAM_SUSPECT",
]:
    pipeline.tracer_cout_selon_seuil(resultats, reels, nom)

---
## 8. Remise

Les deux systèmes classent les 300 courriels du test final. Personne n'a vu
leurs étiquettes. Remettez le fichier `predictions.csv` à l'équipe enseignante.

In [ ]:
test = pipeline.charger_courriels(pipeline.CHEMIN_TEST)
verdicts_test = [
    r["verdict"] for r in scoring_engine.analyser_tous(test, table_marques)
]
predictions_test = ml.predire_tous(test, table, totaux)
print(pipeline.ecrire_predictions(test, verdicts_test, predictions_test))